# 🟣 MODELO 4: Bidirectional LSTM + Layer Normalization

## 📋 Objetivo del ejercicio
Capturar contexto **completo** (pasado Y futuro) usando Bidirectional LSTM.

## ⚠️ Problema del Modelo 3
LSTM solo lee de **izquierda → derecha**. No ve el contexto futuro.

## ✅ Soluciones:
1. **Bidirectional LSTM**: Lee en ambas direcciones
2. **Layer Normalization**: Estabiliza el entrenamiento

### 📚 Teoría de examen:
> "Bidirectional procesa la secuencia en ambas direcciones:
> - Forward: izquierda → derecha
> - Backward: derecha → izquierda
> 
> Concatena ambas salidas para capturar contexto completo.
> Útil cuando el contexto posterior es importante (ej: 'The movie was not ___')"

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)

In [ ]:
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 128  # ⬆️ Aumentamos para modelo más complejo

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(test['message'].values), maxlen=MAX_LEN, padding='post')

print("✅ Datos preparados")

## 🏗️ Modelo Bidirectional LSTM + Layer Normalization

### 📚 ¿Qué es Layer Normalization?
- Normaliza las activaciones de cada capa (media=0, varianza=1)
- **Ventajas**:
  - Estabiliza el entrenamiento
  - Acelera la convergencia
  - Reduce sensibilidad a inicialización
- **Diferencia con Batch Normalization**: 
  - LayerNorm normaliza por features
  - BatchNorm normaliza por batch
  - LayerNorm funciona mejor con RNN/LSTM

### 🔑 Pregunta de examen típica:
**P: ¿Cuántos parámetros tiene Bidirectional LSTM comparado con LSTM normal?**

**R: Aproximadamente el DOBLE. Porque tiene dos LSTMs (forward + backward) 
procesando en paralelo y concatenando las salidas.**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, LayerNormalization

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Dropout(0.3),
    
    # ✨ MEJORA 1: Bidirectional LSTM (lee en ambas direcciones)
    # return_sequences=True porque hay otra capa LSTM después
    Bidirectional(LSTM(64, return_sequences=True)),
    LayerNormalization(),  # ✨ MEJORA 2: Normalización
    Dropout(0.4),
    
    # Segunda capa LSTM (más profundidad)
    Bidirectional(LSTM(32)),  # return_sequences=False (default)
    LayerNormalization(),
    Dropout(0.4),
    
    Dense(32, activation='relu'),
    LayerNormalization(),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### ⚠️ IMPORTANTE: return_sequences

```python
# ✅ CORRECTO:
LSTM(64, return_sequences=True)   # Hay más capas recurrentes después
LSTM(32, return_sequences=False)  # Última capa recurrente

# ❌ ERROR COMÚN EN EXÁMENES:
LSTM(64, return_sequences=False)  # Si hay más LSTM después
LSTM(32, return_sequences=True)   # Si es la última antes de Dense
```

**Shape check:**
- `return_sequences=True` → (batch, timesteps, units)
- `return_sequences=False` → (batch, units)

In [ ]:
# Entrenar
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")
print(f"\n✅ Este debería ser el MEJOR modelo hasta ahora")

In [ ]:
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_bidirectional.csv', index=False)

print("✅ Submission creado: submission_bidirectional.csv")

---

## 📚 Resumen para Examen

### ✅ Ventajas Bidirectional:
- **Contexto completo**: Ve palabras anteriores Y posteriores
- **Mejor comprensión**: Especialmente en frases con negaciones
- Dobla la información capturada

### ❌ Desventajas:
- **Doble de parámetros** → necesita más datos
- **Más lento** de entrenar
- **No puede usarse en tiempo real** (necesita ver toda la secuencia)

### 📊 Layer Normalization:
- Normaliza activaciones
- Estabiliza entrenamiento
- Acelera convergencia

### 🎯 Cuándo usar Bidirectional:
- Clasificación de texto (tienes toda la secuencia)
- Sentimiento, spam, categorización
- Cuando el contexto futuro importa

### 🚫 Cuándo NO usar Bidirectional:
- Predicción en tiempo real (ej: autocompletado)
- Generación de texto
- Series temporales en producción

### 🔄 Alternativa siguiente:
**GRU en lugar de LSTM** para reducir parámetros manteniendo rendimiento.